# SVHN Benchmark: MLP, RNN, CNN on Colab & Kaggle


## How to use
1. **Select accelerator** in your notebook settings (GPU for GPU runs). The notebook can still force **CPU** even on a GPU runtime.
2. Edit the **parameters** in the next cell. For the full factorial at one platform, use `fractions = [0.5, 1.0]` and keep both `cpu` and `gpu` in `hardware_list`.
3. Results append to `kaggle_svhn_benchmark_results.csv`.


In [1]:
# Environment & platform detection
from datetime import datetime, timezone
import os, sys, platform, random, time, subprocess
import numpy as np
import torch

def detect_platform():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ: return 'KAGGLE'
    if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ: return 'COLAB'
    return 'LOCAL'

def get_cpu_name():
    try:
        # Retrieves the specific CPU model name on Linux (Colab/Kaggle)
        return subprocess.check_output("grep -m 1 'model name' /proc/cpuinfo | cut -d: -f2", shell=True).decode().strip()
    except Exception:
        # Fallback for other environments
        return platform.processor() or "Unknown CPU"

PLATFORM = detect_platform()
CPU_NAME = get_cpu_name()
CUDA_AVAILABLE = torch.cuda.is_available()
AUTO_DEVICE = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')

print('Platform:', PLATFORM)
print('CPU Model:', CPU_NAME)
print('CUDA available:', CUDA_AVAILABLE, '| Auto device:', AUTO_DEVICE)
if CUDA_AVAILABLE:
    print('GPU:', torch.cuda.get_device_name(0))

Platform: KAGGLE
CPU Model: Intel(R) Xeon(R) CPU @ 2.00GHz
CUDA available: True | Auto device: cuda
GPU: Tesla T4


In [2]:
# Ensure SciPy (required by torchvision.datasets.SVHN) is available
try:
    import scipy
except Exception:
    print('Installing SciPy...')
    %pip -q install scipy
    import scipy


In [3]:
# ==== Experiment parameters (edit here) ====
fractions = [0.5, 1.0]   # dataset sizes to benchmark (50% vs 100%)
epochs = 5
batch_size = 64
repeats = 3
num_workers = 2
plots = True
save_metrics = 'kaggle_svhn_benchmark_results.csv'
seed = 42
models_to_run = ['mlp', 'rnn', 'cnn']
hardware_list = ['cpu', 'gpu']
out_dir = 'outputs'
data_dir = './data'


In [4]:
# Imports & helpers
import os, csv, math
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def set_seeds(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s);
    torch.cuda.manual_seed_all(s);
    torch.backends.cudnn.deterministic = True;
    torch.backends.cudnn.benchmark = False

class MLPClassifier(nn.Module):
    def __init__(self, input_size=3*32*32, hidden_units=256, num_classes=10):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, hidden_units), nn.ReLU(),
            nn.Linear(hidden_units, hidden_units), nn.ReLU(),
            nn.Linear(hidden_units, num_classes)
        )
    def forward(self, x): return self.model(x)

class RNNClassifier(nn.Module):
    def __init__(self, input_size=96, hidden_units=128, num_layers=1, num_classes=10):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_units, num_layers=num_layers, batch_first=True, nonlinearity='tanh')
        self.fc = nn.Linear(hidden_units, num_classes)
    def forward(self, x):
        x = x.permute(0,2,3,1)              # (B,C,H,W)->(B,H,W,C)
        x = x.reshape(x.size(0), 32, 32*3)  # seq_len=32, features=96
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        return self.fc(last)

class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.drop = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64*8*8, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = torch.flatten(x,1); x = self.drop(x)
        x = F.relu(self.fc1(x)); return self.fc2(x)

def get_transforms():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomGrayscale(p=0.1),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
    ])

def make_datasets(root, dataset_fraction, seed):
    tfm = get_transforms()
    full_train = datasets.SVHN(root=root, split='train', transform=tfm, download=True)
    test_ds = datasets.SVHN(root=root, split='test', transform=tfm, download=True)
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(full_train), generator=g).tolist()
    take = int(len(idx) * dataset_fraction)
    sub = Subset(full_train, idx[:take])
    trn_size = int(0.8 * len(sub))
    val_size = len(sub) - trn_size
    train_ds, val_ds = random_split(sub, [trn_size, val_size], generator=g)
    return train_ds, val_ds, test_ds

def get_loaders(train_ds, val_ds, test_ds, batch_size, device_kind, num_workers):
    pin = (device_kind=='gpu')
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
    return train_loader, val_loader, test_loader

def train_one_epoch(model, device, loader, optimizer, criterion):
    model.train(); run_loss=0.0; correct=0; total=0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward(); optimizer.step()
        run_loss += loss.item()*x.size(0)
        _,pred = torch.max(out,1)
        correct += (pred==y).sum().item(); total += y.size(0)
    return (run_loss/total if total else 0.0), (100.0*correct/total if total else 0.0)

def evaluate(model, device, loader, criterion):
    model.eval(); run_loss=0.0; correct=0; total=0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            run_loss += loss.item()*x.size(0)
            _,pred = torch.max(out,1)
            correct += (pred==y).sum().item(); total += y.size(0)
    return (run_loss/total if total else 0.0), (100.0*correct/total if total else 0.0)

def plot_losses(train_losses, val_losses, out_path):
    plt.figure(figsize=(10,4)); plt.plot(train_losses,label='Train'); plt.plot(val_losses,label='Val');
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.tight_layout(); plt.savefig(out_path); plt.close()

def plot_confusion(model, device, loader, classes, out_path):
    model.eval(); P=[]; T=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device); out=model(x); P.extend(out.argmax(1).cpu().numpy().tolist()); T.extend(y.numpy().tolist())
    cm = confusion_matrix(T,P); disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    fig, ax = plt.subplots(figsize=(6,6)); disp.plot(xticks_rotation='vertical', ax=ax, cmap='Blues', colorbar=False)
    plt.tight_layout(); plt.savefig(out_path); plt.close()


In [5]:
# Benchmark loop: runs (hardware × fractions × models × repeats) and logs CSV
from pathlib import Path
Path(out_dir).mkdir(parents=True, exist_ok=True)
classes = [str(i) for i in range(10)]

header = ['timestamp','platform_hint','device_kind','device_name','torch_version','cuda_available','cuda_version',
          'model','dataset_fraction','epochs','batch_size','train_samples','val_samples','test_samples',
          'repeat_idx','seed','total_train_time_s','per_epoch_time_s_mean','throughput_samples_per_s',
          'final_train_loss','final_train_acc','final_val_loss','final_val_acc','test_loss','test_acc']
write_header = not os.path.exists(save_metrics)
with open(save_metrics, 'a', newline='') as f:
    wr = csv.writer(f)
    if write_header: wr.writerow(header)

    for hw in hardware_list:
        device_kind = 'gpu' if hw=='gpu' and torch.cuda.is_available() else ('cpu' if hw=='cpu' else 'skip')
        if hw=='gpu' and not torch.cuda.is_available():
            print('[SKIP] GPU selected but CUDA not available.')
            continue
        device = torch.device('cuda') if device_kind=='gpu' else torch.device('cpu')
        name = torch.cuda.get_device_name(0) if device_kind=='gpu' else CPU_NAME
        print('=== Hardware: {} ({}) ==='.format(device_kind.upper(), name))

        for frac in fractions:
            set_seeds(seed)
            train_ds, val_ds, test_ds = make_datasets(data_dir, frac, seed)
            train_loader, val_loader, test_loader = get_loaders(train_ds, val_ds, test_ds, batch_size, device_kind, num_workers)

            for model_name in models_to_run:
                for rep in range(1, repeats+1):
                    set_seeds(seed + rep - 1)
                    if model_name=='mlp': model = MLPClassifier().to(device)
                    elif model_name=='rnn': model = RNNClassifier().to(device)
                    elif model_name=='cnn': model = CNNClassifier().to(device)
                    else: raise ValueError(model_name)

                    criterion = nn.CrossEntropyLoss()
                    optim = torch.optim.Adam(model.parameters(), lr=1e-3)
                    epoch_times = []
                    tr_losses, va_losses = [], []

                    t0 = time.perf_counter()
                    for ep in range(1, epochs+1):
                        e0 = time.perf_counter()
                        tr_loss, tr_acc = train_one_epoch(model, device, train_loader, optim, criterion)
                        if device_kind=='gpu': torch.cuda.synchronize()
                        e1 = time.perf_counter(); epoch_times.append(e1-e0)
                        va_loss, va_acc = evaluate(model, device, val_loader, criterion)
                        tr_losses.append(tr_loss); va_losses.append(va_loss)
                        print('[{}][Rep {}] Epoch {}/{} | Train {:.4f}/{:.2f}% | Val {:.4f}/{:.2f}% | Time {:.2f}s'.format(
                              model_name.upper(), rep, ep, epochs, tr_loss, tr_acc, va_loss, va_acc, epoch_times[-1]))

                    if device_kind=='gpu': torch.cuda.synchronize()
                    total_train_time = time.perf_counter() - t0
                    test_loss, test_acc = evaluate(model, device, test_loader, criterion)
                    per_epoch_mean = float(np.mean(epoch_times)) if epoch_times else 0.0
                    throughput = (len(train_loader.dataset)*epochs)/total_train_time if total_train_time>0 else 0.0

                    # Optional plots
                    if plots:
                        base = os.path.join(out_dir, '{}_{}_rep{}_frac{}'.format(model_name, device_kind, rep, frac))
                        plot_losses(tr_losses, va_losses, base + '_loss.png')
                        plot_confusion(model, device, val_loader, classes, base + '_confusion_val.png')

                    wr.writerow([
                        datetime.now(timezone.utc).isoformat(),
                        os.environ.get('KAGGLE_KERNEL_RUN_TYPE','COLAB' if 'COLAB_GPU' in os.environ else 'LOCAL'),
                        device_kind, name, torch.__version__, torch.cuda.is_available(),
                        (torch.version.cuda if torch.version.cuda is not None else ''),
                        model_name, frac, epochs, batch_size, len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset),
                        rep, seed + rep - 1, round(total_train_time,4), round(per_epoch_mean,4), round(throughput,2),
                        round(tr_losses[-1],4), round(tr_acc,2), round(va_losses[-1],4), round(va_acc,2), round(test_loss,4), round(test_acc,2)
                    ])
                    f.flush()
                    print('[DONE] {} | HW {} | Rep {} | Total {:.2f}s | Throughput {:.2f} samples/s | ValAcc {:.2f}% | TestAcc {:.2f}%'.format(
                          model_name.upper(), device_kind, rep, total_train_time, throughput, va_acc, test_acc))

print('Results appended to:', save_metrics)


=== Hardware: CPU (Intel(R) Xeon(R) CPU @ 2.00GHz) ===


100%|██████████| 182M/182M [01:21<00:00, 2.25MB/s] 
100%|██████████| 64.3M/64.3M [00:29<00:00, 2.19MB/s]


[MLP][Rep 1] Epoch 1/5 | Train 1.4478/51.80% | Val 1.0376/67.79% | Time 7.46s
[MLP][Rep 1] Epoch 2/5 | Train 0.9771/69.30% | Val 0.9162/71.10% | Time 7.34s
[MLP][Rep 1] Epoch 3/5 | Train 0.8418/73.87% | Val 0.8706/72.97% | Time 7.51s
[MLP][Rep 1] Epoch 4/5 | Train 0.7548/76.28% | Val 0.7879/76.14% | Time 7.36s
[MLP][Rep 1] Epoch 5/5 | Train 0.6824/78.62% | Val 0.7629/76.58% | Time 7.52s
[DONE] MLP | HW cpu | Rep 1 | Total 45.09s | Throughput 3249.20 samples/s | ValAcc 76.58% | TestAcc 73.08%
[MLP][Rep 2] Epoch 1/5 | Train 1.4330/52.60% | Val 1.0336/67.42% | Time 7.53s
[MLP][Rep 2] Epoch 2/5 | Train 0.9685/70.02% | Val 0.8959/71.98% | Time 7.25s
[MLP][Rep 2] Epoch 3/5 | Train 0.8279/74.30% | Val 0.7967/75.54% | Time 7.83s
[MLP][Rep 2] Epoch 4/5 | Train 0.7532/76.48% | Val 0.7584/76.73% | Time 7.64s
[MLP][Rep 2] Epoch 5/5 | Train 0.6794/78.79% | Val 0.7412/77.33% | Time 7.21s
[DONE] MLP | HW cpu | Rep 2 | Total 45.10s | Throughput 3248.65 samples/s | ValAcc 77.33% | TestAcc 74.86%
[MLP][

In [6]:
# Summarize results: mean±std for time and accuracy, and GPU speedup
import pandas as pd
df = pd.read_csv(save_metrics)
display(df.tail(10))

# Grouped summary
grp = df.groupby(['platform_hint','device_kind','model','dataset_fraction']).agg(
    runs=('repeat_idx','count'),
    total_time_mean=('total_train_time_s','mean'), total_time_std=('total_train_time_s','std'),
    per_epoch_mean=('per_epoch_time_s_mean','mean'),
    throughput_mean=('throughput_samples_per_s','mean'),
    val_acc_mean=('final_val_acc','mean'), val_acc_std=('final_val_acc','std'),
    test_acc_mean=('test_acc','mean'), test_acc_std=('test_acc','std')
).reset_index()
display(grp.sort_values(['model','dataset_fraction','device_kind']))

# Compute GPU speedup vs CPU by matching rows
pivot = grp.pivot_table(index=['platform_hint','model','dataset_fraction'], columns='device_kind', values='total_time_mean')
if 'cpu' in pivot.columns and 'gpu' in pivot.columns:
    pivot['gpu_speedup'] = pivot['cpu'] / pivot['gpu']
    display(pivot)
else:
    print('GPU or CPU results missing; run both hardware configs.')


,timestamp,platform_hint,device_kind,device_name,torch_version,cuda_available,cuda_version,model,dataset_fraction,epochs,...,seed,total_train_time_s,per_epoch_time_s_mean,throughput_samples_per_s,final_train_loss,final_train_acc,final_val_loss,final_val_acc,test_loss,test_acc
26,2026-04-05T00:32:34.550787+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,cnn,0.5,5,...,44,38.6848,6.2056,3787.28,0.3978,88.07,0.4124,88.27,0.4930,86.08
27,2026-04-05T00:34:01.546654+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,mlp,1.0,5,...,42,75.0105,11.9964,3906.45,0.5948,81.72,0.6729,80.33,0.8352,77.04
28,2026-04-05T00:35:24.965723+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,mlp,1.0,5,...,43,74.8780,12.0376,3913.37,0.5965,81.76,0.6708,80.38,0.8391,76.78
29,2026-04-05T00:36:48.165272+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,mlp,1.0,5,...,44,74.6442,11.9902,3925.62,0.5950,81.82,0.6919,79.42,0.8385,77.09
30,2026-04-05T00:38:11.974284+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,rnn,1.0,5,...,42,75.3178,12.0754,3890.51,1.1967,60.29,1.3479,55.66,1.4532,52.09
31,2026-04-05T00:39:35.871185+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,rnn,1.0,5,...,43,75.3294,12.1046,3889.91,1.2204,60.35,1.2545,59.38,1.3016,58.00
32,2026-04-05T00:40:59.087854+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,rnn,1.0,5,...,44,74.6831,11.9719,3923.58,1.1774,61.72,1.2010,60.95,1.2729,58.40
33,2026-04-05T00:42:24.358532+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,cnn,1.0,5,...,42,76.5547,12.3288,3827.66,0.3485,89.51,0.3510,90.00,0.4066,88.10
34,2026-04-05T00:43:49.727266+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,cnn,1.0,5,...,43,76.7640,12.3832,3817.22,0.3329,89.91,0.3377,90.18,0.4017,88.21
35,2026-04-05T00:45:16.208905+00:00,Interactive,gpu,Tesla T4,2.10.0+cu128,True,12.8,cnn,1.0,5,...,44,77.7450,12.4993,3769.05,0.3311,90.05,0.3423,89.84,0.4048,88.03


,platform_hint,device_kind,model,dataset_fraction,runs,total_time_mean,total_time_std,per_epoch_mean,throughput_mean,val_acc_mean,val_acc_std,test_acc_mean,test_acc_std
0,Interactive,cpu,cnn,0.5,3,133.982133,2.698895,23.743667,1093.800000,88.070000,0.294618,86.470000,0.157162
6,Interactive,gpu,cnn,0.5,3,38.742833,0.320317,6.217500,3781.776667,88.113333,0.465224,85.903333,0.546839
1,Interactive,cpu,cnn,1.0,3,273.706000,3.125137,48.865133,1070.680000,89.783333,0.234592,88.383333,0.411987
7,Interactive,gpu,cnn,1.0,3,77.021233,0.635476,12.403767,3804.643333,90.006667,0.170098,88.113333,0.090738
2,Interactive,cpu,mlp,0.5,3,44.660267,0.752903,7.367200,3281.173333,77.330000,0.750000,74.290000,1.048475
8,Interactive,gpu,mlp,0.5,3,36.421700,0.398564,5.813033,4022.930000,77.820000,0.792275,74.120000,1.042881
3,Interactive,cpu,mlp,1.0,3,86.294133,0.476084,14.172333,3395.726667,80.146667,0.607069,76.983333,0.734938
9,Interactive,gpu,mlp,1.0,3,74.844233,0.185470,12.008067,3915.146667,80.043333,0.540401,76.970000,0.166433
4,Interactive,cpu,rnn,0.5,3,48.681500,0.717946,8.063667,3010.003333,52.256667,2.390614,50.700000,2.267884
10,Interactive,gpu,rnn,0.5,3,37.206567,0.112237,5.953167,3937.766667,48.280000,1.528954,46.470000,1.681993


device_kind                                  cpu        gpu  gpu_speedup
platform_hint model dataset_fraction                                    
Interactive   cnn   0.5               133.982133  38.742833     3.458243
                    1.0               273.706000  77.021233     3.553643
              mlp   0.5                44.660267  36.421700     1.226199
                    1.0                86.294133  74.844233     1.152983
              rnn   0.5                48.681500  37.206567     1.308412
                    1.0                97.435200  75.110100     1.297232

---
### Notes
- TorchVision's SVHN loader remaps the digit '0' to label **0** (the original dataset uses **10** for '0'), keeping class labels in **[0..9]** for `CrossEntropyLoss`.
- Loading SVHN `.mat` files via TorchVision requires **SciPy**.
- You can benchmark both **CPU and GPU** in a single GPU-enabled runtime; CPU runs are forced by selecting `device='cpu'`.